In [ ]:
from src.utils.helper import extract_simple_message_from_jsonl

from src.utils.ai_question_generator import AIQuestionGenerator
from prompt_config.config import user_prompt_categories
import os
import pandas as pd

API_KEY = os.getenv('ANTHROPIC_API_KEY')

In [ ]:
df = pd.read_parquet('./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased.parquet')
unique_questions = df.groupby(['title', 'new_question'])['views'].sum().reset_index()

unique_questions = unique_questions.reset_index(drop=True)

# Set id as running number starting from 0
unique_questions['id'] = range(0, len(unique_questions))
unique_questions.to_csv('./data/final/unique_questions_categories.csv', index=False)

In [ ]:
unique_questions.sort_values('views', ascending=True, na_position='last').head()['question'][20481]

In [ ]:
df = pd.read_csv('./data/final/unique_questions_categories.csv')

# Create new id column as row number starting from 1
df['id'] = df.index
df['id'] = df['id'].astype(str)


# df = df.head(10)

generator = AIQuestionGenerator(provider="anthropic"
                                , api_key=API_KEY)

# Prepare content for batch processing
batch_contents = []
for idx, row in df.iterrows():
    content_item = {
        'content': row['new_question'],
        'custom_id': row['id']  # Use URL as custom_id
    }
    batch_contents.append(content_item)


print(f"Creating batch with {len(batch_contents)} requests...")

# Create the batch -- Uncomment when required.
batch_info = generator.generate_qa_batch(
    contents=batch_contents,
    system_prompt="",
    user_prompt=user_prompt_categories,
    max_tokens=64000,
    num_questions=None
)


# Update the base df

In [ ]:
df_base_unique_questions = pd.read_csv('./data/final/unique_questions_categories.csv')
df_categories = extract_simple_message_from_jsonl('./data/final/Categories.jsonl', 'categories')
df_categories['id'] = df_categories['id'].astype(str)
df_base_unique_questions['id'] = df_base_unique_questions['id'].astype(str)
df_base_unique_questions = df_base_unique_questions.merge(df_categories, on='id', how='left')

df_base_unique_questions[df_base_unique_questions['categories'] == 'culture']


In [ ]:
df = pd.read_parquet('./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased.parquet')

df = df.merge(df_base_unique_questions[['new_question', 'categories']], on='new_question', how='left')
df.to_parquet('./data/final/temporal_knowledge_wiki_merged_filtered_pageviews_rephrased_categories.parquet', index=False)